In [ ]:
# Step 1: Download the file
url = "https://neural-nexus-2.s3.ap-south-1.amazonaws.com/Education.zip"
output_path = "/content/Education.zip"

!wget -O {output_path} {url}

# Step 2: Extract the ZIP file
import zipfile
import os

extract_path = "/content/Education"

# Create directory if not exists
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(output_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Download and extraction complete!")

--2026-04-15 07:22:52--  https://neural-nexus-2.s3.ap-south-1.amazonaws.com/Education.zip
Resolving neural-nexus-2.s3.ap-south-1.amazonaws.com (neural-nexus-2.s3.ap-south-1.amazonaws.com)... 3.5.213.168, 3.5.213.231, 3.5.213.9, ...
Connecting to neural-nexus-2.s3.ap-south-1.amazonaws.com (neural-nexus-2.s3.ap-south-1.amazonaws.com)|3.5.213.168|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 20185858205 (19G) [application/zip]
Saving to: ‘/content/Education.zip’

/content/Education. 100%[===================>]  18.80G  29.0MB/s    in 7m 35s  

2026-04-15 07:30:27 (42.3 MB/s) - ‘/content/Education.zip’ saved [20185858205/20185858205]

✅ Download and extraction complete!


In [6]:
!pip install opencv-python torch torchvision tqdm

In [14]:
import os
import cv2
import torch
from torch.utils.data import Dataset
import numpy as np

class VideoDataset(Dataset):
    def __init__(self, root_dir, seq_len=12):
        self.samples = []  # yahan saare video paths + labels store honge
        self.seq_len = seq_len  # kitne frames ek video se lene hai

        self.classes = sorted(os.listdir(root_dir))  # saare class folders
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}  # class → number mapping

        for cls in self.classes:
            cls_path = os.path.join(root_dir, cls)
            for video in os.listdir(cls_path):
                # har video ka path + uska label save kar rahe
                self.samples.append((os.path.join(cls_path, video), self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)  # total videos count

    def load_video(self, path):
        cap = cv2.VideoCapture(path)  # video open
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))  # total frames count

        if total <= 0:  # agar video broken ho
            cap.release()
            return None

        # evenly spaced frames pick karenge
        idxs = np.linspace(0, total-1, self.seq_len).astype(int)

        frames = []
        for i in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)  # direct jump to frame
            ret, frame = cap.read()
            if not ret:
                continue  # agar frame read nahi hua skip karo

            frame = cv2.resize(frame, (112,112))  # size chota = fast
            frames.append(frame)

        cap.release()

        if len(frames) == 0:  # agar koi frame nahi mila
            return None

        # agar frames kam pad gaye to last frame repeat karo
        while len(frames) < self.seq_len:
            frames.append(frames[-1])

        return np.stack(frames)  # list → numpy array

    def __getitem__(self, idx):
        while True:
            path, label = self.samples[idx]  # ek sample uthao
            frames = self.load_video(path)

            if frames is not None:
                # (T,H,W,C) → (T,C,H,W) PyTorch format
                frames = torch.from_numpy(frames).permute(0,3,1,2)
                return frames, label

            # agar video kharab nikla to random dusra pick karo
            idx = np.random.randint(0, len(self.samples))

In [15]:
import torch.nn as nn
import torchvision.models as models
from torchvision.models import ResNet18_Weights

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)  # har timestep ke liye importance score nikalta hai

    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)  # (B,T,1) → kaunsa frame important hai
        context = (weights * x).sum(dim=1)  # important frames ko combine kar rahe
        return context, weights


class CNN_LSTM_Attn(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # CNN: har frame se features nikalta hai
        self.cnn = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        self.cnn.fc = nn.Identity()  # last layer hata di (features chahiye)

        # LSTM: time sequence samajhta hai (frame by frame)
        self.lstm = nn.LSTM(512, 256, num_layers=1, batch_first=True)

        # Attention: important frames pe focus karta hai
        self.attn = Attention(256)

        # Final classifier
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        B,T,C,H,W = x.shape  # batch, time, channels, height, width

        x = x.float() / 255.0  # normalize (0–255 → 0–1)

        # CNN ke liye reshape (saare frames ek saath)
        x = x.view(B*T, C, H, W)
        feat = self.cnn(x)

        # wapas sequence format me lao
        feat = feat.view(B, T, -1)

        # LSTM se temporal info lo
        lstm_out, _ = self.lstm(feat)

        # Attention apply karo
        context, weights = self.attn(lstm_out)
        out = self.fc(context)

        return out, weights

In [ ]:
from torch.utils.data import DataLoader, random_split
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = VideoDataset("/content/Education/Education")

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=1, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=1)

model = CNN_LSTM_Attn(len(dataset.classes)).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

scaler = torch.amp.GradScaler(device="cuda")

train_losses = []
val_accuracies = []

for epoch in range(8):
    model.train()
    total_loss = 0

    for videos, labels in tqdm(train_loader):
        videos = videos.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs, _ = model(videos)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    train_losses.append(total_loss / len(train_loader))

    # VALIDATION
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for videos, labels in val_loader:
            videos = videos.to(device)
            labels = labels.to(device)

            outputs, _ = model(videos)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = correct / total
    val_accuracies.append(acc)

    print(f"Epoch {epoch} | Loss {train_losses[-1]:.4f} | Val Acc {acc:.4f}")

100%|██████████| 13973/13973 [16:35<00:00, 14.04it/s]


Epoch 0 | Loss 2.1076 | Val Acc 0.3419


100%|██████████| 13973/13973 [15:44<00:00, 14.79it/s]


Epoch 1 | Loss 1.8248 | Val Acc 0.3502


100%|██████████| 13973/13973 [15:50<00:00, 14.70it/s]


...

100%|██████████| 13973/13973 [15:56<00:00, 14.61it/s]


Epoch 3 | Loss 1.5617 | Val Acc 0.3726


100%|██████████| 13973/13973 [16:00<00:00, 14.56it/s]


Epoch 4 | Loss 1.4673 | Val Acc 0.3872


 80%|████████  | 11188/13973 [12:49<02:46, 16.77it/s]

In [ ]:
plt.plot(train_losses, label="Loss")
plt.plot(val_accuracies, label="Val Accuracy")
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

model.eval()

videos, _ = next(iter(val_loader))
videos = videos.to(device)

_, weights = model(videos)

weights = weights[0].detach().cpu().numpy()

plt.imshow(weights.T, cmap='hot', aspect='auto')
plt.title("Attention over Frames")
plt.xlabel("Frame Index")
plt.ylabel("Importance")
plt.colorbar()
plt.show()